## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = pd.read_excel("RealEstate_HousePrice_Dataset_4200.xlsx")
data.head()

## 2. Understand the data

In [ ]:
print("Rows and columns:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())

print("\nMissing values:")
print(data.isnull().sum())

print("\nDuplicate rows:", data.duplicated().sum())
data.describe().round(2)

In [ ]:
data = data.drop_duplicates()
data = data.dropna(subset=["house_price_inr"])

y = data["house_price_inr"]
X = data.drop(columns=["house_id", "house_price_inr"])

X = X.fillna(X.median(numeric_only=True))

print("Features:", list(X.columns))
print("Target: house_price_inr")

## 3. Explore the data

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(data["house_price_inr"], bins=30, edgecolor="black")
plt.title("House Price Distribution")
plt.xlabel("House price (INR)")
plt.ylabel("Number of houses")
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(data.select_dtypes(include="number").corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Between Numeric Columns")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(data["area_sqft"], data["house_price_inr"], alpha=0.5)
plt.title("House Area and Price")
plt.xlabel("Area (sq.ft.)")
plt.ylabel("Price (INR)")
plt.show()

## 4. Training and testing data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## Part C — Simple linear regression

In [ ]:
X_area = data[["area_sqft"]]
y_price = data["house_price_inr"]

area_train, area_test, price_train, price_test = train_test_split(
    X_area, y_price, test_size=0.20, random_state=42
)

simple_model = LinearRegression()
simple_model.fit(area_train, price_train)
simple_prediction = simple_model.predict(area_test)

print("Intercept:", round(simple_model.intercept_, 2))
print("Slope:", round(simple_model.coef_[0], 2))

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(area_test["area_sqft"], price_test, alpha=0.5, label="Actual prices")

sorted_rows = np.argsort(area_test["area_sqft"].values)
plt.plot(area_test["area_sqft"].values[sorted_rows],
         simple_prediction[sorted_rows], color="red", label="Regression line")

plt.xlabel("Area (sq.ft.)")
plt.ylabel("Price (INR)")
plt.title("Simple Linear Regression")
plt.legend()
plt.show()

### Check the residuals

In [ ]:
residuals = price_test - simple_prediction

plt.figure(figsize=(7, 4))
plt.scatter(simple_prediction, residuals, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted price")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()

## Part D — Evaluation metrics

In [ ]:
def check_model(actual, predicted, feature_count):
    mse = mean_squared_error(actual, predicted)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, predicted)
    n = len(actual)

    if n > feature_count + 1:
        adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_count - 1)
    else:
        adjusted_r2 = np.nan

    return {
        "MSE": mse,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Adjusted R2": adjusted_r2
    }

simple_result = check_model(price_test, simple_prediction, 1)
pd.DataFrame([simple_result], index=["Simple Linear Regression"]).T

## Part E — Multiple linear regression

In [ ]:
multiple_model = LinearRegression()
multiple_model.fit(X_train, y_train)
multiple_prediction = multiple_model.predict(X_test)

multiple_result = check_model(y_test, multiple_prediction, X_test.shape[1])

results = pd.DataFrame(
    [simple_result, multiple_result],
    index=["Simple Linear Regression", "Multiple Linear Regression"]
)
results

In [ ]:
coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": multiple_model.coef_
})
coefficients.sort_values("Coefficient", key=abs, ascending=False)

## Part F — Polynomial regression

In [ ]:
poly_results = []
poly_models = {}

for degree in [2, 3]:
    polynomial = PolynomialFeatures(degree=degree, include_bias=False)
    area_train_poly = polynomial.fit_transform(area_train)
    area_test_poly = polynomial.transform(area_test)

    model = LinearRegression()
    model.fit(area_train_poly, price_train)
    prediction = model.predict(area_test_poly)
    poly_models[degree] = (polynomial, model)

    score = check_model(price_test, prediction, degree)
    score["Train RMSE"] = np.sqrt(mean_squared_error(
        price_train, model.predict(area_train_poly)
    ))
    poly_results.append({"Model": f"Polynomial degree {degree}", **score})

pd.DataFrame(poly_results).set_index("Model")

In [ ]:
area_values = np.linspace(area_train["area_sqft"].min(),
                          area_train["area_sqft"].max(), 250).reshape(-1, 1)

plt.figure(figsize=(8, 5))
plt.scatter(area_test["area_sqft"], price_test, alpha=0.3, label="Test data")
plt.plot(area_values, simple_model.predict(area_values), label="Linear")

for degree, (polynomial, model) in poly_models.items():
    plt.plot(area_values, model.predict(polynomial.transform(area_values)),
             label=f"Polynomial degree {degree}")

plt.xlabel("Area (sq.ft.)")
plt.ylabel("Price (INR)")
plt.title("Linear and Polynomial Models")
plt.legend()
plt.show()

## Part G — Gradient descent

In [ ]:
scale_x = StandardScaler()
X_train_scaled = scale_x.fit_transform(X_train)
X_test_scaled = scale_x.transform(X_test)

y_mean = y_train.mean()
y_std = y_train.std()
y_train_scaled = (y_train.to_numpy() - y_mean) / y_std

def run_gradient_descent(X, y, method, epochs=150, learning_rate=0.01, batch_size=32):
    rng = np.random.default_rng(42)
    X = np.c_[np.ones(len(X)), X]
    weights = np.zeros(X.shape[1])
    loss_history = []

    for epoch in range(epochs):
        if method == "batch":
            error = X @ weights - y
            weights -= learning_rate * (2 / len(y)) * (X.T @ error)
        else:
            order = rng.permutation(len(y))
            step = 1 if method == "stochastic" else batch_size

            for start in range(0, len(y), step):
                ids = order[start:start + step]
                error = X[ids] @ weights - y[ids]
                weights -= learning_rate * (2 / len(ids)) * (X[ids].T @ error)

        loss_history.append(np.mean((X @ weights - y) ** 2))

    return weights, loss_history

def predict_gradient(X, weights):
    X = np.c_[np.ones(len(X)), X]
    return X @ weights

gradient_results = {}

for method in ["batch", "stochastic", "mini-batch"]:
    weights, losses = run_gradient_descent(
        X_train_scaled, y_train_scaled, method
    )
    predicted_scaled = predict_gradient(X_test_scaled, weights)
    predicted_price = predicted_scaled * y_std + y_mean

    gradient_results[method] = {
        "weights": weights,
        "losses": losses,
        "prediction": predicted_price
    }

    print(method, "test RMSE:",
          round(np.sqrt(mean_squared_error(y_test, predicted_price)), 2))

In [ ]:
plt.figure(figsize=(7, 4))
for method, values in gradient_results.items():
    plt.plot(values["losses"], label=method)

plt.xlabel("Epoch")
plt.ylabel("Training MSE (scaled target)")
plt.title("Gradient Descent Loss")
plt.legend()
plt.show()

## Part H — Bias, variance and model diagnostics


In [ ]:
diagnostic_rows = []

diagnostic_rows.append({
    "Model": "Simple Linear",
    "Train RMSE": np.sqrt(mean_squared_error(
        price_train, simple_model.predict(area_train)
    )),
    "Test RMSE": np.sqrt(mean_squared_error(price_test, simple_prediction))
})

diagnostic_rows.append({
    "Model": "Multiple Linear",
    "Train RMSE": np.sqrt(mean_squared_error(
        y_train, multiple_model.predict(X_train)
    )),
    "Test RMSE": np.sqrt(mean_squared_error(y_test, multiple_prediction))
})

for degree, (polynomial, model) in poly_models.items():
    diagnostic_rows.append({
        "Model": f"Polynomial degree {degree}",
        "Train RMSE": np.sqrt(mean_squared_error(
            price_train, model.predict(polynomial.transform(area_train))
        )),
        "Test RMSE": np.sqrt(mean_squared_error(
            price_test, model.predict(polynomial.transform(area_test))
        ))
    })

diagnostics = pd.DataFrame(diagnostic_rows).set_index("Model")
display(diagnostics)

diagnostics.plot(kind="bar", figsize=(8, 4))
plt.ylabel("RMSE (INR)")
plt.title("Training and Testing Error")
plt.xticks(rotation=15)
plt.show()

## Part I — Final comparison and conclusion

In [ ]:
comparison = [
    {"Model": "Simple Linear Regression", **simple_result},
    {"Model": "Multiple Linear Regression", **multiple_result}
]

for degree, (polynomial, model) in poly_models.items():
    prediction = model.predict(polynomial.transform(area_test))
    comparison.append({
        "Model": f"Polynomial degree {degree}",
        **check_model(price_test, prediction, degree)
    })

comparison = pd.DataFrame(comparison).set_index("Model")
display(comparison.sort_values("RMSE"))

best_name = comparison["RMSE"].idxmin()
print("Best model by test RMSE:", best_name)
print("Test RMSE:", round(comparison.loc[best_name, "RMSE"], 2))
print("Test MAE:", round(comparison.loc[best_name, "MAE"], 2))
print("Test R2:", round(comparison.loc[best_name, "R2"], 4))